In [ ]:
dbutils.widgets.removeAll()

dbutils.widgets.text("env", "")
dbutils.widgets.text("dt", "")

env = dbutils.widgets.get("env")
dt = dbutils.widgets.get("dt")

print(f'run for: {env} {dt}')

import pandas as pd
import numpy as np
import os

import duckdb

from apo_config.config import config

from common.utils.logutil import setuplogger
from common.utils import dtutil, sqlutil

from data_access.spark.api import sparkmanager
import pyspark.sql.functions as sparkfuncs

from data_access.utils import tosqllist

from dash_reports.issuer_debt_in_index.data.fx import adjust_fx_amt_fields
from dash_reports.issuer_debt_in_index.data.factset import adjust_fx_financials
from data_access.dbtables import BOND_REFERENCEDATA_APP

from dash_reports.issuer_debt_in_index.data import debt

# setuplogger()

config.init(env or os.environ.get('ENV'))

read_cache = (dt != '')

In [ ]:
#Add these two widgets + this block right after config.init(...):

In [ ]:
dbutils.widgets.removeAll()

dbutils.widgets.text("env", "")
dbutils.widgets.text("dt", "")
dbutils.widgets.text("catalog", "my_catalog")     # <-- NEW
dbutils.widgets.text("schema", "my_schema")       # <-- NEW

env = dbutils.widgets.get("env")
dt = dbutils.widgets.get("dt")
catalog = dbutils.widgets.get("catalog")          # <-- NEW
schema = dbutils.widgets.get("schema")            # <-- NEW

print(f'run for: {env} {dt} {catalog}.{schema}')

import pandas as pd
import numpy as np
import os

import duckdb

from apo_config.config import config

from common.utils.logutil import setuplogger
from common.utils import dtutil, sqlutil

from data_access.spark.api import sparkmanager
import pyspark.sql.functions as sparkfuncs

from data_access.utils import tosqllist

from dash_reports.issuer_debt_in_index.data.fx import adjust_fx_amt_fields
from dash_reports.issuer_debt_in_index.data.factset import adjust_fx_financials
from data_access.dbtables import BOND_REFERENCEDATA_APP

from dash_reports.issuer_debt_in_index.data import debt

# setuplogger()

config.init(env or os.environ.get('ENV'))

# --- NEW: route unqualified table names + stub external API calls ---
from synthetic_debt_universe.run_local_workspace import configure_workspace, apply_stubs
configure_workspace(catalog=catalog, schema=schema)
apply_stubs()
# ----------------------------------------------------------------------

read_cache = (dt != '')

### debt df from index data

In [ ]:
dt = dtutil.todate( dt or dtutil.dateadd(dtutil.today(), '-1b') )

In [ ]:
debt_df = debt.get_debt_df(dt, calc_pecs_flag=False, build_issuer_curves=True, use_spark=False, merge_dcs_fs=False)

### adjust issuer amt using bbg fx

In [ ]:
from data_access.dbtables.market_data.spot import fx

In [ ]:
fxdf = fx.get_fx(dtutil.dateadd(dt, '-2y'), dt)

In [ ]:
debt_df = debt_df.drop(columns=[x for x in debt_df.columns if 'fx' in x])

debt_df = adjust_fx_amt_fields(debt_df, fxdf, amt_cols=['amt_outstanding_loc', 'amt_issued_loc', 'eq_mkt_cap_mm'])
fx_spot_df = fxdf[['ccy', 'asof_date', 'value']].sort_values(['asof_date', 'ccy'])

fx_spot_df.columns = ['currency', 'asof_date', 'fx_rate']

fx_spot_df['asof_date'] = pd.to_datetime(fx_spot_df['asof_date']).dt.date

debt_df = adjust_fx_financials(debt_df, fx_spot_df.dropna(subset=['fx_rate'], how='all'))

### size filters on debt df

In [ ]:
debt_df['unclassified'] = np.where(debt_df['instrument_type'] == 'Fixed Income/Unclassified', 1, 0)

In [ ]:
debt_df['amt_outstanding_debt_only_usd'] = np.where(debt_df['instrument_type'] == 'Equity', 0, debt_df['amt_outstanding_usd'])
debt_df['amt_outstanding_debt_only_loc'] = np.where(debt_df['instrument_type'] == 'Equity', 0, debt_df['amt_outstanding_loc'])

In [ ]:
non_cds_rows = (debt_df['instrument_type'] != 'CDS') & (debt_df['unclassified'] == 0)

for iss_col in ['issuer_amt_outstanding_debt_only_usd', 'issuer_amt_outstanding_usd']:
    debt_df.loc[:, iss_col] = 0

    debt_df.loc[non_cds_rows, iss_col] = \
        debt_df[non_cds_rows].groupby('ticker')[iss_col.replace('issuer_', '')].transform(sum)

    debt_df.loc[:, iss_col] = \
        debt_df[:].groupby('ticker')[iss_col].transform(max)

In [ ]:
min_issuer_amt_out = 1000
min_issue_size = 300

debt_df['issuer_min_amt_out_flag'] = (debt_df['issuer_amt_outstanding_debt_only_usd'] >= min_issuer_amt_out).astype(int)
debt_df['issue_min_size_flag'] = (debt_df['amt_outstanding_debt_only_usd'] >= min_issue_size).astype(int)

debt_df['issuer_min_amt_out_flag'] = debt_df.groupby('ticker')['issuer_min_amt_out_flag'].transform(lambda x: 1 if x.any() else 0)
debt_df['issue_min_size_flag'] = debt_df.groupby('ticker')['issue_min_size_flag'].transform(lambda x: 1 if x.any() else 0)

In [ ]:
debt_df.shape

### deriv some new metrics for curves and weights for aggregation logic on returns, yields, spreads, duration, prices

In [ ]:
debt_df['unsecured_curve_spread_2Y5Y'] = debt_df['unsecured_curve_spread_5Y'] - debt_df['unsecured_curve_spread_2Y']
debt_df['unsecured_curve_spread_5Y10Y'] = debt_df['unsecured_curve_spread_10Y'] - debt_df['unsecured_curve_spread_5Y']
debt_df['sec_vs_unsec_5Y_spread'] = debt_df['unsecured_curve_spread_5Y'] - debt_df['secured_curve_spread_5Y']

In [ ]:
#sorted(debt_df.columns.tolist())

In [ ]:
return_cols = ['bond_point_dtd_return', 'bond_point_1w_return', 'bond_point_mtd_return', 'bond_point_1m_return', 'bond_point_qtd_return', 'bond_point_ytd_return']

In [ ]:
for i, x in enumerate(return_cols):
    if i != 0:
        debt_df[x] = debt_df[x].replace(0, np.nan).combine_first(debt_df[return_cols[i-1]])

In [ ]:
debt_df['mv_usd'] = debt_df['amt_outstanding_debt_only_usd'].mul(debt_df['price']).mul(0.01)

In [ ]:
debt_df['CS01_usd'] = debt_df['spread_duration'].mul(0.0001).mul(debt_df['amt_outstanding_debt_only_usd']).mul(debt_df['price']).mul(0.01)

In [ ]:
# List of columns to check for zero values
issuer_col_fields_zero_check = [
    #'bond_point_ytd_return',
    'yield_to_worst_conv',
    'oas',
    'spread_duration',
    'effective_duration'
]
# Replace null values with zero
debt_df[issuer_col_fields_zero_check] = debt_df[issuer_col_fields_zero_check].fillna(0)
# Filter rows where any of the specified columns are zero
#filtered_debt_df = debt_df[(debt_df[issuer_col_fields_zero_check] != 0).all(axis=1) & (debt_df['unclassified'] == 0)]
# Filter rows where all specified columns are non-zero and no column has negative values
filtered_debt_df = debt_df[
    # (debt_df[issuer_col_fields_zero_check] != 0).all(axis=1) &
    (debt_df[issuer_col_fields_zero_check] > 0).all(axis=1) &
    (debt_df['unclassified'] == 0)
]

In [ ]:
ticker_sec_sum = filtered_debt_df.groupby(['ticker', 'secured_unsecured']).agg({'amt_outstanding_debt_only_usd': 'sum', 'CS01_usd': 'sum', 'mv_usd': 'sum'}).reset_index()
ticker_sum = filtered_debt_df.groupby(['ticker']).agg({'amt_outstanding_debt_only_usd': 'sum', 'CS01_usd': 'sum', 'mv_usd': 'sum'}).reset_index()

In [ ]:
#ticker_sum.head()

In [ ]:
debt_df_issuer = pd.merge(debt_df, ticker_sec_sum, how='left', on=['ticker', 'secured_unsecured'], suffixes=('', '_sec_unsec'))

debt_df_issuer = pd.merge(debt_df_issuer, ticker_sum, how='left', on=['ticker'], suffixes=('', '_ticker'))

In [ ]:
#debt_df_issuer.columns.sort_values().tolist()

In [ ]:
filtered_df = debt_df_issuer[
    #(debt_df_issuer[issuer_col_fields_zero_check] != 0).all(axis=1) &
    (debt_df_issuer[issuer_col_fields_zero_check] > 0).all(axis=1) &
    (debt_df_issuer['unclassified'] == 0)]

In [ ]:
# Calculate the weights for most fields using amt outstanding in usd as weights
debt_df_issuer.loc[filtered_df.index, 'amt_outstanding_usd_sec_unsec_wght'] = filtered_df['amt_outstanding_debt_only_usd'] / filtered_df['amt_outstanding_debt_only_usd_sec_unsec']
debt_df_issuer.loc[filtered_df.index, 'amt_outstanding_usd_ticker_wght'] = filtered_df['amt_outstanding_debt_only_usd'] / filtered_df['amt_outstanding_debt_only_usd_ticker']

# calc weights for spreads using spread duration as weights
debt_df_issuer.loc[filtered_df.index, 'spread_duration_usd_sec_unsec_wght'] = filtered_df['CS01_usd'] / filtered_df['CS01_usd_sec_unsec']
debt_df_issuer.loc[filtered_df.index, 'spread_duration_usd_ticker_wght'] = filtered_df['CS01_usd'] / filtered_df['CS01_usd_ticker']

debt_df_issuer.loc[filtered_df.index, 'mv_usd_sec_unsec_wght'] = filtered_df['mv_usd'] / filtered_df['mv_usd_sec_unsec']
debt_df_issuer.loc[filtered_df.index, 'mv_usd_ticker_wght'] = filtered_df['mv_usd'] / filtered_df['mv_usd_ticker']

debt_df_issuer['instrument_desc'] = np.where(debt_df_issuer['instrument_type'] == 'Equity', debt_df_issuer['instrument_id'] + " Equity", debt_df_issuer['instrument_desc'])

In [ ]:
issuer_col_fields = ['price', 'bond_point_dtd_return', 'bond_point_1w_return', 'bond_point_1m_return', 'bond_point_mtd_return', 'bond_point_ytd_return',]
issuer_col_fields_spd_wght = ['oas', 'yield_to_worst_conv', ]
issuer_col_fields_mv_wght = ['spread_duration', 'effective_duration', ]

In [ ]:
# Calculate the weighted average for each column by dividing the sum of the weighted values by the sum of 'amt_outstanding_usd' at ticker level
for col in issuer_col_fields:
    debt_df_issuer[col + '_ticker_avg'] = debt_df_issuer[col] * debt_df_issuer['amt_outstanding_usd_ticker_wght']

# Calculate the weighted average for each column by dividing the sum of the weighted values by the sum of 'amt_outstanding_usd' at secured/unsecured level
for col in issuer_col_fields:
    debt_df_issuer[col + '_sec_unsec_ticker_avg'] = debt_df_issuer[col] * debt_df_issuer['amt_outstanding_usd_sec_unsec_wght']

In [ ]:
# Calculate the weighted average for each column by dividing the sum of the weighted values by the sum of 'cs01' at ticker level
for col in issuer_col_fields_spd_wght:
    debt_df_issuer[col + '_ticker_avg'] = debt_df_issuer[col] * debt_df_issuer['spread_duration_usd_ticker_wght']

# Calculate the weighted average for each column by dividing the sum of the weighted values by the sum of 'cs01' at secured/unsecured level
for col in issuer_col_fields_spd_wght:
    debt_df_issuer[col + '_sec_unsec_ticker_avg'] = debt_df_issuer[col] * debt_df_issuer['spread_duration_usd_sec_unsec_wght']

In [ ]:
# Calculate the weighted average for each column by dividing the sum of the weighted values by the sum of 'cs01' at ticker level
for col in issuer_col_fields_mv_wght:
    debt_df_issuer[col + '_ticker_avg'] = debt_df_issuer[col] * debt_df_issuer['mv_usd_ticker_wght']

# Calculate the weighted average for each column by dividing the sum of the weighted values by the sum of 'cs01' at secured/unsecured level
for col in issuer_col_fields_mv_wght:
    debt_df_issuer[col + '_sec_unsec_ticker_avg'] = debt_df_issuer[col] * debt_df_issuer['mv_usd_sec_unsec_wght']

In [ ]:
ticker = 'CCL'
debt_df_issuer[debt_df_issuer['ticker'] == ticker].groupby(['ticker', 'secured_unsecured', 'seniority', 'instrument_type', 'instrument_id', 'instrument_desc'], dropna=False)[[
    'amt_outstanding_usd', 'amt_outstanding_loc', 'unsecured_curve_spread_5y', 'secured_curve_spread_5y', 'issuer_amt_outstanding_usd',
    'bond_point_ytd_return_ticker_avg', 'price_sec_unsec_ticker_avg', 'yield_to_worst_conv_ticker_avg',
    'oas_ticker_avg', 'spread_duration_ticker_avg', 'effective_duration_ticker_avg']]\
    .agg({'amt_outstanding_usd': 'sum', 'amt_outstanding_loc': 'sum', 'unsecured_curve_spread_5y': 'mean', 'secured_curve_spread_5y': 'mean', 'issuer_amt_outstanding_usd': 'max',
          'bond_point_ytd_return_ticker_avg': 'sum', 'price_sec_unsec_ticker_avg': 'sum', 'yield_to_worst_conv_ticker_avg': 'sum', 'oas_ticker_avg': 'sum', 'spread_duration_ticker_avg': 'sum',
          'effective_duration_ticker_avg': 'sum'})

### adding additional ratios

In [ ]:
debt_df_issuer['FF_NET_DEBT_2_EBITDA'] = debt_df_issuer['FF_NET_DEBT'] / debt_df_issuer['FF_EBITDA_OPER']
debt_df_issuer['FF_NET_DEBT_2_EBITDA_MINUS_CAPEX'] = debt_df_issuer['FF_NET_DEBT'] / (debt_df_issuer['FF_EBITDA_OPER'] - debt_df_issuer['FF_CAPEX'])
debt_df_issuer['FF_DEBT_2_EV'] = debt_df_issuer['FF_DEBT'] / debt_df_issuer['FF_ENTRPR_VAL']

debt_df_issuer['FF_EBITDA_2_INT_EXP_NET'] = debt_df_issuer['FF_EBITDA_OPER'] / debt_df_issuer['FF_INT_EXP_NET']

In [ ]:
debt_df_issuer.shape

In [ ]:
unsec_wide_constant = 600
sec_wide_constant = 500

sec_v_unsec_wide_constant = 400
sec_v_unsec_tight_constant = 150

debt_df_issuer['unsecured_wide'] = (debt_df_issuer['unsecured_curve_spread_5y'] >= unsec_wide_constant).astype(int)
debt_df_issuer['secured_wide'] = (debt_df_issuer['secured_curve_spread_2y'] >= sec_wide_constant).astype(int)

In [ ]:
debt_df_issuer['unsecured_wide_flag'] = debt_df_issuer.groupby('ticker')['unsecured_wide'].transform(lambda x: 1 if x.any() else 0)
debt_df_issuer['secured_wide_flag'] = debt_df_issuer.groupby('ticker')['secured_wide'].transform(lambda x: 1 if x.any() else 0)

In [ ]:
issuer_sec_unsec_spds = debt_df_issuer.groupby(['ticker', 'secured_unsecured'])['oas_sec_unsec_ticker_avg'].sum().unstack('secured_unsecured')

In [ ]:
debt_df_issuer.groupby(['ticker', 'secured_unsecured']).agg({'oas_sec_unsec_ticker_avg': 'sum', 'sec_vs_unsec_5Y_spread': 'mean'}).unstack('secured_unsecured')

In [ ]:
issuer_sec_unsec_spds['sec_unsec_issuer_spd'] = np.where(
    (issuer_sec_unsec_spds['SEC'].fillna(0) != 0) & (issuer_sec_unsec_spds['UNSEC'].fillna(0) != 0),
    issuer_sec_unsec_spds['UNSEC'] - issuer_sec_unsec_spds['SEC'],
    np.nan
)

In [ ]:
# Map the new column 'is_difference_greater_200' to be 1 if 'difference' > 200, else 0
issuer_sec_unsec_spds['secured_vs_unsecured_wide'] = (issuer_sec_unsec_spds['sec_unsec_issuer_spd'] > sec_v_unsec_wide_constant).astype(int)
issuer_sec_unsec_spds['secured_vs_unsecured_tight'] = (issuer_sec_unsec_spds['sec_unsec_issuer_spd'] < sec_v_unsec_tight_constant).astype(int)

In [ ]:
issuer_debt_1m_return = debt_df_issuer.groupby('ticker')['bond_point_1m_return_ticker_avg'].sum()
issuer_debt_1m_return = issuer_debt_1m_return.rename('1m_bpt_return_byIssuer')
debt_df_issuer = pd.merge(debt_df_issuer, issuer_debt_1m_return.reset_index(), on=['ticker'], how='left')

# Step 1: Check the condition within each 'ticker' group
# If any row meets the condition in the group, mark the entire group
condition = (
    debt_df_issuer.groupby('ticker')['1m_bpt_return_byIssuer'].transform(lambda x: (x > 0).any()) &
    debt_df_issuer.groupby('ticker')['one_month_pct'].transform(lambda x: (x < 0).any())
)

# Step 2: Assign the flag to the new column based on the condition
debt_df_issuer['eq_down_debt_up_flag'] = condition.astype(int)

In [ ]:
debt_df_issuer2 = pd.merge(debt_df_issuer.drop(columns=['unsecured_wide', 'secured_wide', '1m_bpt_return_byIssuer']), issuer_sec_unsec_spds[['secured_vs_unsecured_wide', 'secured_vs_unsecured_tight', 'sec_unsec_issuer_spd']], on=['ticker'], how='left')

In [ ]:
# Replace 0 in 'price' with 100
debt_df_issuer2['price'] = debt_df_issuer2['price'].fillna(100)

In [ ]:
duckdb.sql("select asof_date, unclassified, ticker, secured_unsecured, instrument_id, source, merge_type, instrument_type, instrument_desc, amt_outstanding_usd, oas, sec_unsec_issuer_spd, FF_EBITDA_OPER, FF_EBITDA_OPER_USD from debt_df_issuer2 where ticker='VSTIET'")

In [ ]:
debt_df_issuer3 = debt_df_issuer2

In [ ]:
debt_df_issuer3['amt_outstanding_mkt_usd'] = np.where(debt_df_issuer3['instrument_type'] == 'Equity', 1, debt_df_issuer3['price'] * 0.01) * debt_df_issuer3['amt_outstanding_usd']
debt_df_issuer3['FF_EBITDA_OPER'] = debt_df_issuer3['FF_EBITDA_OPER'].fillna(debt_df_issuer3.groupby('ticker')['FF_EBITDA_OPER'].transform('mean'))

# book leverage
debt_df_issuer3['mult_book'] = debt_df_issuer3['amt_outstanding_usd'] / debt_df_issuer3['FF_EBITDA_OPER']
debt_df_issuer3['mult_mkt'] = debt_df_issuer3['amt_outstanding_mkt_usd'] / debt_df_issuer3['FF_EBITDA_OPER']

#debt_df_issuer3['unclassified'] = np.where(debt_df_issuer3['instrument_type'] == 'Fixed Income/Unclassified', 1, 0)

secured_debt_ratio = debt_df_issuer3[(debt_df_issuer3['secured_unsecured'] == 'SEC') & (debt_df_issuer3['unclassified'] == 0)].groupby('ticker')[['mult_book', 'mult_mkt']].sum()

unsecured_debt_ratio = debt_df_issuer3[(debt_df_issuer3['secured_unsecured'].isin(['SEC', 'UNSEC'])) & (debt_df_issuer3['unclassified'] == 0)].groupby('ticker')[['mult_book', 'mult_mkt']].sum()
sub_debt_ratio = debt_df_issuer3[(debt_df_issuer3['secured_unsecured'].isin(['SEC', 'UNSEC', 'SUB'])) & (debt_df_issuer3['unclassified'] == 0)].groupby('ticker')[['mult_book', 'mult_mkt']].sum()

equity_ratio = debt_df_issuer3[(debt_df_issuer3['unclassified'] == 0)].groupby('ticker')[['mult_book', 'mult_mkt']].sum()

In [ ]:
debt_df_issuer3 = debt_df_issuer3.merge(secured_debt_ratio, on='ticker', how='left', suffixes=('', '_secured_sum'))

In [ ]:
debt_df_issuer3 = debt_df_issuer3.merge(unsecured_debt_ratio, on='ticker', how='left', suffixes=('', '_unsecured_sum'))

debt_df_issuer3 = debt_df_issuer3.merge(sub_debt_ratio, on='ticker', how='left', suffixes=('', '_sub_sum'))

debt_df_issuer3 = debt_df_issuer3.merge(equity_ratio, on='ticker', how='left', suffixes=('', '_equity_sum'))

conditions = [
    debt_df_issuer3['secured_unsecured'] == 'SEC',
    debt_df_issuer3['secured_unsecured'] == 'UNSEC',
    debt_df_issuer3['secured_unsecured'] == 'SUB',
    debt_df_issuer3['secured_unsecured'] == 'Equity'
]

In [ ]:
choices_book = [
    debt_df_issuer3['mult_book_secured_sum'],
    debt_df_issuer3['mult_book_unsecured_sum'],
    debt_df_issuer3['mult_book_sub_sum'],
    debt_df_issuer3['mult_book_equity_sum']
]

choices_mkt = [
    debt_df_issuer3['mult_mkt_secured_sum'],
    debt_df_issuer3['mult_mkt_unsecured_sum'],
    debt_df_issuer3['mult_mkt_sub_sum'],
    debt_df_issuer3['mult_mkt_equity_sum']
]

debt_df_issuer3['ebitda_mult_book_consol'] = np.select(conditions, choices_book, default=np.nan)
debt_df_issuer3['ebitda_mult_mkt_consol'] = np.select(conditions, choices_mkt, default=np.nan)

debt_df_issuer3 = debt_df_issuer3.drop(columns=['mult_book_secured_sum', 'mult_mkt_secured_sum', 'mult_book_unsecured_sum', 'mult_mkt_unsecured_sum', 'mult_book_sub_sum', 'mult_mkt_sub_sum'])

In [ ]:
duckdb.sql("select asof_date, unclassified, ticker, secured_unsecured, instrument_id, source, merge_type, instrument_type, instrument_desc, amt_outstanding_usd, oas, FF_EBITDA_OPER, FF_EBITDA_OPER_USD from debt_df_issuer3 where ticker='VSTIET'")

### adding dealogic data

In [ ]:
related_instr_sql = """
SELECT t2.linked_instrument_id, t2.instrument_id, t1.bbg_current_{0}_list
FROM (
    SELECT
        instrtbl.instrument_id,
        ARRAY_JOIN(SORT_ARRAY(COLLECT_SET(CAST(instrtbl.linked_instrument_id AS STRING))), ',') AS bbg_current_{0}_list
    FROM
        related_instruments instrtbl
    WHERE
        id_type = '{0}'
    GROUP BY
        instrtbl.instrument_id
    HAVING
        SIZE(COLLECT_SET(instrtbl.linked_instrument_id)) >= 1
) t1
JOIN related_instruments t2
ON t1.instrument_id = t2.instrument_id;
"""

related_ids_sql = """
select * from v_related_ids
"""

related_instr = sqlutil.readdfsql(related_instr_sql.format('isin'), app=BOND_REFERENCEDATA_APP)

cusip_related_instr = sqlutil.readdfsql(related_instr_sql.format('cusip'), app=BOND_REFERENCEDATA_APP)

related_ids = sqlutil.readdfsql(related_ids_sql, app=BOND_REFERENCEDATA_APP)

In [ ]:
related_instr.shape, cusip_related_instr.shape, related_ids.shape

In [ ]:
if read_cache and os.path.exists('dealogic_df2.parquet'):
    dealogic_df2 = pd.read_parquet('dealogic_df2.parquet')
else:
    dealogic_bond_sql = '''select * from dbo.v_dealogic'''

    dealogic_df_raw = sqlutil.readdfsql(dealogic_bond_sql, app='databricks')

    dealogic_df_raw.shape

    dealogic_df2 = pd.merge(dealogic_df_raw, related_instr[['instrument_id', 'linked_instrument_id']],
                             how='left', left_on='primary_isin', right_on='linked_instrument_id')

    dealogic_df2.drop(columns=['linked_instrument_id'], inplace=True)
    dealogic_df2 = dealogic_df2.rename(columns={'instrument_id': 'instrument_id_dealogic'})

    dealogic_df2 = pd.merge(dealogic_df2, cusip_related_instr[['instrument_id', 'linked_instrument_id']],
                             how='left', left_on='cusip', right_on='linked_instrument_id')

    dealogic_df2.drop(columns=['linked_instrument_id'], inplace=True)

    dealogic_df2 = dealogic_df2.rename(columns={'instrument_id': 'instrument_id_dealogic_cusip'})

    dealogic_df2 = pd.merge(dealogic_df2,
                             related_ids[['instrument_id_isin', 'instrument_id_cusip']]\
                                 .sort_values(['instrument_id_isin', 'instrument_id_cusip'])\
                                 .drop_duplicates(subset=['instrument_id_cusip'], keep='first'),
                             how='left', left_on='instrument_id_dealogic_cusip', right_on='instrument_id_cusip')

    dealogic_df2['instrument_id_dealogic'] = dealogic_df2['instrument_id_dealogic'].combine_first(dealogic_df2['instrument_id_isin'])

    dealogic_df2.head()

In [ ]:
from data_access.dbtables.factset.dcs import get_isin_ticker_df

In [ ]:
debtdf_ticker_isins = get_isin_ticker_df(dtutil.dateadd(dt, '-1y'), ticker_regions=debt_df_issuer3['ticker_region'].unique().tolist())

In [ ]:
debtdf_ticker_isins = pd.merge(debtdf_ticker_isins, related_instr, left_on=['isin'], right_on=['linked_instrument_id'], how='left')

In [ ]:
ticker_region_map = debt_df_issuer3[['ticker_region', 'ticker', 'issuer_name']].groupby(['ticker_region', 'ticker']).count()\
    .sort_values('issuer_name').reset_index()\
    .drop_duplicates(subset=['ticker_region', 'ticker'], keep='last')\
    .drop(columns='issuer_name')

In [ ]:
debtdf_ticker_isins = pd.merge(debtdf_ticker_isins, ticker_region_map, left_on=['ticker_region'], right_on=['ticker_region'], how='left')

In [ ]:
debtdf_ticker_isins['instrument_id'] = debtdf_ticker_isins['instrument_id'].combine_first(debtdf_ticker_isins['isin'])

match_cols_dealogic = ['instrument_id_dealogic']

match_cols_debt = ['instrument_id']

dealogic_issuer_ticker_map = pd.merge(debtdf_ticker_isins[match_cols_debt + ['ticker']].dropna().drop_duplicates(subset=match_cols_debt),
                                       dealogic_df2[match_cols_dealogic + ['issuer_name']].dropna().drop_duplicates(subset=match_cols_dealogic),
                                       how='outer', left_on=match_cols_debt, right_on=match_cols_dealogic)

In [ ]:
dealogic_issuer_ticker_map = dealogic_issuer_ticker_map.groupby(['issuer_name', 'ticker'])[match_cols_dealogic].count()\
    .sort_values(match_cols_dealogic).reset_index()\
    .drop_duplicates(subset='issuer_name', keep='last')\
    .drop(columns=match_cols_dealogic)

dealogic_df2 = pd.merge(dealogic_df2, dealogic_issuer_ticker_map, on='issuer_name', how='left')

In [ ]:
debt_df_issuer3['current_coupon'] = debt_df_issuer3['current_coupon'].combine_first(pd.to_numeric(debt_df_issuer3['coupon'], errors='coerce'))

In [ ]:
inst_id_uniq_from_ticker = debt_df_issuer3[['ticker', 'current_coupon', 'maturity', 'instrument_id_uniq']].dropna().drop_duplicates()

dealogic_df2 = pd.merge(dealogic_df2, inst_id_uniq_from_ticker,
                         left_on=['ticker', 'coupon_percent', 'maturity'],
                         right_on=['ticker', 'current_coupon', 'maturity'],
                         how='left').drop(columns=['current_coupon'])

dealogic_df2['instrument_id_dealogic'] = dealogic_df2['instrument_id_dealogic'].combine_first(dealogic_df2['instrument_id_uniq'])

In [ ]:
debt_df_issuer3[debt_df_issuer3['issuer_name'].fillna('').str.contains('Mars Inc')][['issuer_name', 'ticker_region', 'source']]
#[['issuer_name', 'ticker', 'ticker_parent', 'instrument_id_dealogic', 'cusip', 'coupon_percent', 'maturity', 'years_to_maturity', 'pricing_date']]\
#.sort_values('pricing_date', ascending=False)

In [ ]:
import re
import inflect
#import fractions

engine = inflect.engine()

def remove_sentence_starting_with_word(paragraph, word):
    # Replace newline characters with spaces
    paragraph = paragraph.replace('\n', ' ').replace('"', '')

    # Split the paragraph into sentences
    sentences = re.split(r'(?<=[.!?]) +', paragraph)

    # Remove sentences that start with the word
    sentences = [sentence for sentence in sentences if not sentence.lstrip().startswith(word)]

    # Join the remaining sentences back into a paragraph
    new_paragraph = ' '.join(sentences)

    return new_paragraph

def extract_lien_type(paragraph):
    # Define a dictionary to map ordinal number suffixes to their numbers
    ordinal_word_map = {
        1: "first",
        2: "second",
        3: "third",
        4: "fourth",
        5: "fifth",
        6: "sixth",
    }

    ordinal_number_map = {i: engine.ordinal(i) for i in range(1, 11)}

    # Create reverse dictionaries to map ordinal words and numbers back to their original keys
    reverse_ordinal_word_map = {v: k for k, v in ordinal_word_map.items()}
    #reverse_ordinal_word_map.update({'secured': 1})

    reverse_ordinal_number_map = {v: k for k, v in ordinal_number_map.items()}

    # Convert the paragraph to lower case and split it into words, remove non alpha-numeric characters
    words = re.sub(r'[^a-zA-Z0-9]+', ' ', paragraph).lower().split()

    # Iterate over each word in the paragraph

In [ ]:
    for i in range(len(words)-1):
        # If a word matches a key in the dictionary and the next word is "lien", construct the lien type
        if words[i] in reverse_ordinal_word_map and np.any([words[i+j].startswith("lien") for j in range(7) if (i+j) < len(words)]):
            return str(reverse_ordinal_word_map[words[i]]) + " " + "Lien"
        if words[i] in reverse_ordinal_number_map and np.any([words[i+j].startswith("lien") for j in range(7) if (i+j) < len(words)]):
            return str(reverse_ordinal_number_map[words[i]]) + " " + "Lien"

        # If a word matches a key in the dictionary and the next word is "secured", construct the lien type
        if words[i] in reverse_ordinal_word_map and np.any([words[i+j].startswith("secured") for j in range(7) if (i+j) < len(words)]):
            return str(reverse_ordinal_word_map[words[i]]) + " " + "Lien"

        if words[i] in reverse_ordinal_number_map and np.any([words[i+j].startswith("secured") for j in range(7) if (i+j) < len(words)]):
            return str(reverse_ordinal_number_map[words[i]]) + " " + "Lien"

        # Check for decimal numbers in the format "x.y" followed by "lien"
        if re.match(r'\d+\.\d+', words[i]) and np.any([words[i+j].startswith("lien") for j in range(7) if (i+j) < len(words)]):
            return words[i] + " Lien"

    for i in range(len(words)-1):
        if words[i] == "secured" and words[i+1] in ["by", "on", "with"]:
            return "secured"

    # If no lien type is found, return None
    return None

In [ ]:
def extract_gty(paragraph):
    # Convert the paragraph to lower case and split it into words, remove non alpha-numeric characters
    words = re.sub(r'[^a-zA-Z0-9]+', ' ', paragraph).lower().split()

    for i in range(len(words)-1):
        if words[i].startswith("guarant") and words[i+1] in ["by", "on", "with"]:
            return 1
    return None

def extract_toggle(paragraph):
    # Convert the paragraph to lower case and split it into words, remove non alpha-numeric characters
    words = re.sub(r'[^a-zA-Z0-9]+', ' ', paragraph).lower().split()

    for i in range(len(words)-1):
        if words[i].startswith("toggle"):
            return 1
    return None

def extract_bailin(paragraph):
    # Convert the paragraph to lower case and split it into words, remove non alpha-numeric characters
    words = re.sub(r'[^a-zA-Z0-9]+', ' ', paragraph).lower().split()

    for i in range(len(words)-1):
        if words[i] == "bail" and words[i+1] == "in":
            return 1
    return None

def extract_private(paragraph):
    # Convert the paragraph to lower case and split it into words, remove non alpha-numeric characters
    paragraph = re.sub(r'[^a-zA-Z0-9]+', ' ', paragraph).lower()

    # Define the pattern to match the phrases
    pattern = r'\b(private placement|privately placed|private offering)\b'

    # Search for the pattern in the paragraph (case insensitive)
    if re.search(pattern, paragraph, re.IGNORECASE):
        return 1
    return None

In [ ]:
dealogic_df2['comment_modified'] = dealogic_df2['comment'].apply(remove_sentence_starting_with_word, word="Proceeds")
dealogic_df2['lien'] = dealogic_df2['comment_modified'].apply(extract_lien_type)

dealogic_df2['has_guaranty'] = dealogic_df2['comment_modified'].apply(extract_gty)

dealogic_df2['has_toggle'] = dealogic_df2['comment_modified'].apply(extract_toggle)

dealogic_df2['has_bail_in'] = dealogic_df2['comment_modified'].apply(extract_bailin)

dealogic_df2['has_private'] = dealogic_df2['comment_modified'].apply(extract_private)

In [ ]:
#sorted(debt_df_issuer3.columns.tolist())

In [ ]:
use_HY_only = True

In [ ]:
# Fill null values in the index column with 'NULL'
#debt_df_issuer3['index'] = debt_df_issuer3['index'].fillna('NULL')

# Group by ticker and index and count occurrences
grouped_df = debt_df_issuer3[debt_df_issuer3['instrument_type'] != 'CDS'].groupby(['ticker', 'index'])['index'].count().reset_index(name='count')

# Identify tickers that have counts only in 'CDX', 'ER00', 'UR00'
def filter_tickers(group):
    indices = set(group['index'])
    # Exclude tickers that have counts only in 'CDX', 'ER00', 'UR00'
    return not indices.issubset({'CDX', 'ER00', 'UR00'})

# Apply the filter to remove tickers with counts only in the specified indices, but keep tickers with a blank index
filtered_tickers = grouped_df.groupby('ticker').filter(filter_tickers)

In [ ]:
# if use_HY_only:
#     debt_df_issuer4 = debt_df_issuer3[debt_df_issuer3['ticker'].isin(filtered_tickers['ticker'].unique())]
# else:
debt_df_issuer4 = debt_df_issuer3

In [ ]:
[x for x in debt_df_issuer4.columns if 'eq' in x]

In [ ]:
# debt_df_issuer4 = debt_df_issuer4[all_cols]

In [ ]:
#filter out deals that have matured
dealogic_df2['maturity'] = pd.to_datetime(dealogic_df2['maturity'], errors='coerce').dt.date
dealogic_df3 = dealogic_df2[dealogic_df2['maturity'] > dt]

In [ ]:
dealogic_df2[dealogic_df2['dealogic_issuer_company_id'] == 575664]

In [ ]:
dealogic_df3.groupby(['dealogic_issuer_company_id', 'maturity', 'coupon_percent'])['deal_id'].count().sort_values(ascending=False)

In [ ]:
dealogic_df3.drop_duplicates(subset=['dealogic_issuer_company_id', 'maturity', 'coupon_percent', 'instrument_id_dealogic']).groupby('instrument_id_dealogic')['deal_id'].count().sort_values(ascending=False)

In [ ]:
duckdb.sql('''select coupon_percent, maturity, primary_isin, instrument_id_dealogic, * from dealogic_df3 where instrument_id_dealogic = 'DEAL0LB640A4' ''').df()

In [ ]:
level_1_summary_cols_vals = ['issuer_amt_outstanding_usd', 'amt_outstanding_usd', 'amt_outstanding_loc', \
    'one_week_pct', 'bond_point_1w_return_ticker_avg', 'bond_point_1w_return', 'bond_point_alpha_1w_return', 'bond_point_beta_1w_return',\
    'mtd_pct', 'bond_point_mtd_return_ticker_avg', 'bond_point_mtd_return', 'bond_point_alpha_mtd_return', 'bond_point_beta_mtd_return',\
    'one_month_pct', 'bond_point_1m_return_ticker_avg', 'bond_point_1m_return', 'bond_point_alpha_1m_return', 'bond_point_beta_1m_return', \
    'ytd_pct', 'bond_point_ytd_return_ticker_avg', 'bond_point_ytd_return', 'bond_point_beta_ytd_return', 'bond_point_alpha_ytd_return',\
    'one_day_pct', 'bond_point_dtd_return_ticker_avg', 'bond_point_dtd_return', 'bond_point_alpha_dtd_return', 'bond_point_beta_dtd_return',\
    'unsecured_curve_spread_2Y5Y', 'sec_vs_unsec_5Y_spread', 'unsecured_curve_spread_5Y', 'unsecured_curve_spread_5Y10Y', 'secured_curve_spread_2Y', 'price', 'price_ticker_avg',
    'price_sec_unsec_ticker_avg', 'three_month_volatility', 'yield_to_worst_conv_ticker_avg', 'yield_to_worst_conv_sec_unsec_ticker_avg', \
    'maturity_years', 'sec_unsec_issuer_spd', 'oas', 'oas_ticker_avg', 'oas_sec_unsec_ticker_avg', 'spread_duration_ticker_avg', 'effective_duration_ticker_avg', 'spread_duration',
    'effective_duration', \
    'FF_DEBT_2_EV', 'FF_EBITDA_OPER', 'FF_EBITDA_OPER_USD', 'FF_EBITDA_OPER_findox', 'FF_EBITDA_OPER_factset', 'FF_NET_DEBT_2_EBITDA', 'FF_NET_DEBT', 'FF_NET_DEBT_findox',
    'FF_NET_DEBT_factset', 'FF_DEBT', 'FF_DEBT_findox', 'FF_DEBT_factset', 'FF_INT_EXP_NET', 'FF_INT_EXP_NET_findox', 'FF_INT_EXP_NET_factset',
    'FF_SALES', 'FF_SALES_findox', 'FF_SALES_factset', 'FF_ENTRPR_VAL', 'FF_CAPEX', 'FF_CAPEX_findox', 'FF_CAPEX_factset', 'ebitda_mult_book_consol', 'ebitda_mult_mkt_consol',
    'FF_NET_DEBT_2_EBITDA_MINUS_CAPEX', 'FF_EBITDA_2_INT_EXP_NET', \
    'amt_outstanding_usd_sec_unsec_wght', 'amt_outstanding_usd_ticker_wght',
    'spread_duration_usd_sec_unsec_wght', 'spread_duration_usd_ticker_wght',
    'mv_usd_sec_unsec_wght', 'mv_usd_ticker_wght',
    'eq_mkt_cap_mm_usd', 'amt_outstanding_debt_only_usd', 'amt_outstanding_debt_only_loc', 'current_coupon', 'maturity',
    'default_prob', 'liquidity_score', 'actual_cds_bond_basis_mid']

level_1_summary_cols_filter = ['unclassified', 'loan_only_main_issuer', 'index', 'issue_min_size_flag', 'issuer_min_amt_out_flag', 'unsecured_wide_flag', 'secured_wide_flag', 'secured_vs_unsecured_wide',
    'secured_vs_unsecured_tight', 'eq_down_debt_up_flag', 'broad_industry', 'is_restricted', 'source']

level_1_summary_cols_category = ['ticker', 'index', 'issuer_name', 'narrow_industry', 'secured_unsecured', 'seniority', 'instrument_type', 'instrument_desc', 'instrument_id', 'instrument_id_uniq',
    'ticker_region', 'currency_financial', 'bbg_id', 'id_bb_company', 'merge_type']

dealogic_issuer_cols = ['instrument_id_dealogic', 'currency', 'maturity', 'is_issuer_negative_pledge', 'governing_law', 'issue_type', 'debt_ranking', 'security_type', 'lien', 'has_guaranty', 'has_toggle',
    'has_make_whole_call', 'is_hybrid_capital', 'sponsor_activity_type', 'primary_financial_sponsor_immediate_parent_id', 'primary_financial_sponsor_company_parent_id']

dealogic_primary_trade_cols = ['announcement_date', 'pricing_date', 'settlement_date', 'primary_financial_sponsor_name', 'current_financial_sponsors', 'tranche_value_local', 'tranche_value_usd', 'issue_price',
    'issue_spread', 'has_private', 'tranche_bank_net_revenue_usd', 'primary_fee', 'deal_id', 'use_of_proceeds', 'lead_left_tranche_bank', 'comment']

all_cols = level_1_summary_cols_vals + level_1_summary_cols_filter + level_1_summary_cols_category

#group all primary trades to deal with add-ons
def concat_strings(series):
    return '; '.join(series.dropna().astype(str).unique())

#issuer columns
dealogic_df3_drop_dups = dealogic_df3[dealogic_issuer_cols].sort_values(dealogic_issuer_cols).drop_duplicates('instrument_id_dealogic', keep='first')

# Perform the group by and apply the aggregation function on all_cols
dealogic_df4 = dealogic_df3.groupby('instrument_id_dealogic').agg({col: concat_strings for col in dealogic_primary_trade_cols}).reset_index()

In [ ]:
dealogic_df4 = pd.merge(dealogic_df3_drop_dups, dealogic_df4, how='inner', on='instrument_id_dealogic').drop_duplicates('instrument_id_dealogic', keep='first')

dealogic_df4.columns = dealogic_df4.columns.map(lambda x: x.lower())

dealogic_df3_drop_dups.shape, dealogic_df4.shape

In [ ]:
dealogic_df4[dealogic_df4['instrument_id_dealogic'] == 'US83001TAB82']

### left join debt issuer data with dealogic meta data

In [ ]:
debt_df_issuer5 = pd.merge(debt_df_issuer4, dealogic_df4.dropna(subset=['instrument_id_dealogic']), how='left', left_on='instrument_id_uniq', right_on='instrument_id_dealogic', suffixes=('', '_dealogic'))

### investigate which deals do not have an instrument id like an isin

In [ ]:
debt_df_issuer5.shape

In [ ]:
duckdb.sql("select unclassified, ticker, broad_industry, secured_unsecured, instrument_id, source, merge_type, instrument_type, instrument_desc, price, oas_ticker_avg, sec_unsec_issuer_spd, amt_outstanding_usd, oas, FF_EBITDA_OPER, FF_EBITDA_OPER_USD from debt_df_issuer5 where ticker='KKR'")

In [ ]:
debt_df_issuer5.shape

In [ ]:
from data_access.spark.azureblob import BlobDownloader
from common.utils.fileutil import TempFolder
# blob_dloader = BlobDownloader()

In [ ]:
import re
from datetime import datetime
import os

# def read_repo_df_from_blob():
#     folder = 'data/uploads/short_univ_repo_data'

#     pattern = r"DM Top 500 IG HY.*(\d{8})\."
#     files = blob_dloader.is_files(folder)

#     files_dt = sorted([x for x in files if re.search(pattern, x)])[-1]

#     with TempFolder() as tempdir:
#         file_path = os.path.join(tempdir, files_dt)
#         blob_dloader.download(f'{folder}/{files_dt}', file_path)

#         excel_data = pd.read_excel(file_path, sheet_name=None)
#         dataframes = []
#         for sheet_name in excel_data.keys():
#             df = pd.read_excel(file_path, sheet_name=sheet_name)
#             dataframes.append(df)

#         repo_df = pd.concat(dataframes, ignore_index=True)
#         asof_date_str = re.search(pattern, files_dt).group(1)
#         asof_date = datetime.strptime(asof_date_str, '%Y%m%d').strftime('%Y-%m-%d')

#         repo_df['repo_asof_date'] = pd.to_datetime(asof_date)

#     return repo_df

import re
from datetime import datetime
import os

def read_repo_df_from_volume(volume_path):
    pattern = r"DM Top 500 IG HY.*(\d{8})\."
    files = sorted([f for f in os.listdir(volume_path) if re.search(pattern, f)])
    assert len(files) > 0, f"no matching repo file found in: {volume_path}"
    files_dt = files[-1]
    file_path = os.path.join(volume_path, files_dt)

    excel_data = pd.read_excel(file_path, sheet_name=None)
    dataframes = [pd.read_excel(file_path, sheet_name=sheet_name) for sheet_name in excel_data.keys()]

    repo_df = pd.concat(dataframes, ignore_index=True)
    asof_date_str = re.search(pattern, files_dt).group(1)
    asof_date = datetime.strptime(asof_date_str, '%Y%m%d').strftime('%Y-%m-%d')
    repo_df['repo_asof_date'] = pd.to_datetime(asof_date)

    return repo_df

In [ ]:
from data_importer.usd_api.helpers import to_naming_convention_string
def process_col(x):
    col_map = {}
    x = x.replace('(%)', '(pct)')
    x = to_naming_convention_string(x)
    x = col_map.get(x, x)
    return x

In [ ]:
# repo_df = read_repo_df_from_blob()

REPO_VOLUME_PATH = f"/Volumes/{catalog}/{schema}/short_univ_repo_data"   # adjust volume name if different
repo_df = read_repo_df_from_volume(REPO_VOLUME_PATH)

In [ ]:
repo_df.columns = [process_col(x) for x in repo_df.columns.tolist()]

repo_df2 = pd.merge(repo_df, related_instr[['instrument_id', 'linked_instrument_id']],
                     how='left', left_on='isin', right_on='linked_instrument_id')

repo_df2.drop(columns=['linked_instrument_id'], inplace=True)

repo_df2 = repo_df2.rename(columns={'instrument_id': 'instrument_id_repo'})

repo_df2 = repo_df2.dropna(subset=['instrument_id_repo'])

#removing duplicates that show up on multiple tabs
repo_df2 = repo_df2.drop_duplicates(subset=['instrument_id_repo'])

repo_df2.shape

In [ ]:
duckdb.sql("""
select instrument_id_repo, borrow_base, * from repo_df2
where security_description like 'W00%'
""").df()

In [ ]:
repo_df2 = repo_df2.rename(columns=lambda x: x.replace('queterly', 'quarterly'))

In [ ]:
reqd_repo_columns = ['repo_asof_date', 'instrument_id_repo', 'borrow_base', 'weekly_delta', 'monthly_delta', 'quarterly_delta',
    'change_in_monthly_borrow_base', 'change_in_quarterly_borrow_base', 'specials_ac',
    'borrow_base_ratio_borrow_base_amount_outstanding', 'amount_outstanding']

existing_repo_columns = []

for x in reqd_repo_columns:
    if x in repo_df2.columns:
        existing_repo_columns.append(x)

repo_df3 = repo_df2[existing_repo_columns]

In [ ]:
borrow_base_history_repo_cols = ['borrow_base1_week_back', 'borrow_base1_month_back', 'borrow_base3_month_back']

repo_df3['borrow_base1_week_back'] = repo_df3['borrow_base'] - repo_df3['weekly_delta']
repo_df3['borrow_base1_month_back'] = repo_df3['borrow_base'] - repo_df3['monthly_delta']
repo_df3['borrow_base3_month_back'] = repo_df3['borrow_base'] - repo_df3['quarterly_delta']
repo_df3['amount_outstanding'] = pd.to_numeric(repo_df3['amount_outstanding'], errors='coerce')

In [ ]:
debt_df_issuer6 = pd.merge(debt_df_issuer5,
                            repo_df3,
                            how='left', left_on='instrument_id_uniq', right_on='instrument_id_repo')

In [ ]:
repo_rows = pd.notnull(debt_df_issuer6['instrument_id_repo'])
debt_df_issuer6.loc[repo_rows, 'amount_outstanding'] = debt_df_issuer6.loc[repo_rows, 'amount_outstanding'].combine_first(debt_df_issuer6.loc[repo_rows, 'amt_outstanding_loc'])
debt_df_issuer6.loc[repo_rows, 'borrow_base_ratio_borrow_base_amount_outstanding'] = \
    debt_df_issuer6.loc[repo_rows, 'borrow_base'] / debt_df_issuer6.loc[repo_rows, 'amount_outstanding']

In [ ]:
debt_df_issuer6['broad_industry'] = debt_df_issuer6['broad_industry'].str.title()
debt_df_issuer6['knowledge_date'] = debt_df_issuer6['asof_date']
debt_df_issuer6['asof_date'] = dt
debt_df_issuer6['load_date'] = datetime.now()

In [ ]:
# Create a mapping dictionary
industry_mapping = {
    'Automotive': 'Automotive',
    'Banking': 'Banking and Financial Services',
    'Financial': 'Banking and Financial Services',
    'Financial Services': 'Banking and Financial Services',
    'Insurance': 'Banking and Financial Services',
    'Basic Industry': 'Basic Materials',
    'Chemicals': 'Basic Materials',
    'Metals And Mining': 'Basic Materials',
    'Paper And Packaging': 'Basic Materials',
    'Consumer Goods': 'Consumer Discretionary',
    'Consumer Products': 'Consumer Discretionary',
    'Retail': 'Consumer Discretionary',
    'Leisure': 'Consumer Discretionary',
    'Gaming Lodging And Leisure': 'Consumer Discretionary',
    'Energy': 'Energy',
    'Healthcare': 'Healthcare',
    'Capital Goods': 'Industrials',
    'Industrials': 'Industrials',
    'Transportation': 'Industrials',
    'Broadcasting': 'Media and Entertainment',
    'Cable and Satellite': 'Media and Entertainment',
    'Media': 'Media and Entertainment',
    'Diversified Media': 'Media and Entertainment',
    'Real Estate': 'Real Estate',
    'Housing': 'Real Estate',
    'Technology': 'Technology',
    'Utility': 'Utilities',
    'Food And Beverages': 'Food and Beverages',
    'Services': 'Services'
}

# Map the broad_industry column to new categories
debt_df_issuer6['broad_industry_consol'] = debt_df_issuer5['broad_industry'].map(industry_mapping)

sparkmanager.init()
# sqlutil.execsql(f" delete from debt_features where asof_date='{dt}' ", app='databricks')---
sqlutil.execsql(f" delete from {catalog}.{schema}.debt_features where asof_date='{dt}' ", app='databricks')

In [ ]:
# sparkmanager.pandas_to_sql(debt_df_issuer6, 'debt_features', app='databricks')
sparkmanager.pandas_to_sql(debt_df_issuer6, f'{catalog}.{schema}.debt_features', app='databricks')

In [ ]:
from credit_strategy.pbi_access.pbi_api import PBIAPI
from credit_strategy.pbi_access.const import PBI_MOVERS_REPORT_DATASET_ID, PBI_INDUSTRY_SCREEN_DATASET_ID, PBI_GOVT_SPENDING_DOGE_DATASET_ID

if dt == dtutil.dateadd(dtutil.today(), '-1b'):
    pbi_api = PBIAPI()

    pbi_api.refresh_dataset(PBI_MOVERS_REPORT_DATASET_ID, dependencies=[PBI_MOVERS_REPORT_DATASET_ID])
    pbi_api.refresh_dataset(PBI_INDUSTRY_SCREEN_DATASET_ID, dependencies=[PBI_INDUSTRY_SCREEN_DATASET_ID])

    pbi_api.refresh_dataset(PBI_GOVT_SPENDING_DOGE_DATASET_ID, dependencies=[PBI_GOVT_SPENDING_DOGE_DATASET_ID])

In [ ]:
len(debt_df_issuer6[debt_df_issuer6['instrument_id_repo'].notna()])